# FANT 3 — CERN-inspired feature tests (2026-04-21)

Exercises the 11 physics-derived features landed this session on a Colab A100 (or any GPU). All tests are opt-in: default behavior is preserved, so this notebook simply turns each flag on, asserts the diagnostic fires, and restores defaults.

Feature ↔ CERN reference:

| # | Feature | File | CERN CDS |
|---|---|---|---|
| 1 | Ritort replicon eigenvalue | `matryoshka_moe.py` | 263665 |
| 2 | P(q) two-replica overlap | `spinor_apollonian.py` | 782816 |
| 3 | Bell CHSH correlator | `spinor_apollonian.py` | 111654 |
| 4 | MoR Apollonian retrieval wiring | `fant3_model.py` | — |
| 5 | Adaptive-K (AdS radial) | `recursion.py` | 387573 |
| 6 | RDT K-extrapolation override | `recursion.py` | — |
| 7 | ISRM contractive decay | `recursion.py` | arxiv:2507.10524 |
| 8 | Area-law attention window | `attention.py` | 2199026 |
| 9 | Litim compact-support scheduler | `training/schedulers.py` | 492334 |
| 10 | Diag-EF Fisher preconditioner | `training/optim.py` | 2752417 |
| 11 | FSS scale-ladder collapse | `scripts/fss_collapse.py` | 207748 |


## 0 · Environment setup

Upload `fant_code.zip` to your Drive (anywhere under `MyDrive/`). The first cell mounts Drive, finds the zip, unzips to `/content/fant2/`, and sets the working directory.

In [ ]:
# Cell 0.1 — Drive mount + locate zip
import os, sys, shutil, zipfile, glob
IN_COLAB = 'COLAB_GPU' in os.environ or 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    zip_candidates = glob.glob('/content/drive/MyDrive/**/fant_code.zip', recursive=True)
    assert zip_candidates, 'Upload fant_code.zip somewhere under MyDrive first.'
    ZIP_PATH = zip_candidates[0]
    WORK_DIR = '/content/fant2'
    if os.path.isdir(WORK_DIR):
        shutil.rmtree(WORK_DIR)
    os.makedirs(WORK_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(WORK_DIR)
    print('unzipped', ZIP_PATH, '->', WORK_DIR)
else:
    # Local fallback (Windows/Linux): assume cwd is already the repo root.
    WORK_DIR = os.getcwd()
os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)
print('cwd =', os.getcwd())

In [ ]:
# Cell 0.2 — deps (Colab has torch/numpy; only scipy may be needed)
try:
    import scipy  # noqa: F401
except Exception:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'scipy'])
import torch, numpy as np
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| device', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

In [ ]:
# Cell 0.3 — import smoke
from fant3.config import fant3_50m
from fant3.model.matryoshka_moe import MatryoshkaRouter, MatryoshkaMoEFFN
from fant3.model.spinor_apollonian import SpinorApollonianMemory
from fant3.model.fant3_model import FANT3Model
from fant3.model.recursion import MoRShared
from fant3.model.attention import MASAAttention
from fant3.training import schedule_multiplier, cosine_schedule, litim_schedule, precondition_router_grads_
print('all imports OK')

## 1 · Ritort replicon eigenvalue (AT-line predictor)

`MatryoshkaRouter.compute_replicon(logits)` returns `Var(top1 − top2) − (T·cap)²`. Positive on a bimodal pre-collapse distribution; ≤0 when routing is stable.

In [ ]:
stable   = torch.randn(1000, 8) * 0.5
bimodal  = torch.zeros(1000, 8)
bimodal[:500, 0] = 5.0                 # half confident
bimodal[500:] = torch.randn(500, 8) * 0.1  # half uncertain
R_stable  = float(MatryoshkaRouter.compute_replicon(stable))
R_bimodal = float(MatryoshkaRouter.compute_replicon(bimodal))
print(f'stable  replicon = {R_stable:+.3f}  (expect ≤ 0)')
print(f'bimodal replicon = {R_bimodal:+.3f}  (expect > 0, flags collapse risk)')
assert R_bimodal > R_stable, 'bimodal should score higher than stable'
print('PASS')

## 2 · P(q) two-replica overlap + Bell CHSH correlator

Both wired into `SpinorApollonianMemory.get_stats()`.
- `pq_overlap_*` — Berg-Billoire-Janke CDS 782816.
- `chsh_S` — Bell CDS 111654. Collapsed packs give S=√2; empty give 0.

In [ ]:
mem = SpinorApollonianMemory(dim=64, alpha_cap=200, beta_cap=200)
torch.manual_seed(0)
# Synthesize pre-RMSnorm hiddens split along axis 0 so chirality ~= 0.5
a_h = torch.randn(120, 64) + torch.tensor([3.0] + [0.0]*63)
b_h = torch.randn(120, 64) + torch.tensor([-3.0] + [0.0]*63)
with torch.no_grad():
    mem.proj_spinor.weight.zero_()
    mem.proj_spinor.weight[1, 0] = 1.0
mem.store(a_h, hidden_preRMSnorm=a_h)
mem.store(b_h, hidden_preRMSnorm=b_h)
s = mem.get_stats()
print({k: (f'{v:.3f}' if isinstance(v, float) else v) for k,v in s.items()})
assert abs(s['chirality_balance'] - 0.5) < 0.1, 'packs should be balanced'
assert 'pq_bimodality' in s and 'chsh_S' in s
print('PASS')

## 3 · MoR Apollonian retrieval wiring

`FANT3Model._retrieve_for_mor()` reads from `self.memory`, softmax-weights top-k, and feeds the result into `MoRShared.forward(retrieved=...)`. Fires only when MoR.LTI + Apollonian channel are both enabled.

In [ ]:
cfg = fant3_50m()
cfg.mor_lti_injection_enabled = True
cfg.mor_lti_apollonian_channel = True
cfg.spinor_apollonian_enabled  = True
m = FANT3Model(cfg).eval()
ids = torch.randint(0, cfg.vocab_size, (2, 32))
# Empty memory — retrieval returns None
with torch.no_grad():
    x = m.tok_emb(ids)
    for blk in m.dense_blocks: x = blk(x)
    r_empty = m._retrieve_for_mor(x)
print('empty-memory retrieve -> ', r_empty)
# Populate then retry
with torch.no_grad():
    _ = m(ids, targets=ids, store_to_memory=True)
    r_full = m._retrieve_for_mor(x)
print(f'after store: memory items={int(m.memory.alpha_count)+int(m.memory.beta_count)} retrieved shape={tuple(r_full.shape)}')
assert r_empty is None and r_full.shape == (2, 32, cfg.dim)
print('PASS')

## 4 · Adaptive-K (AdS radial depth)

`cfg.mor_adaptive_depth = True` clamps recursion to `min(max_depth, ⌊log₂ T⌋+1)`. For typical T≥4 it’s a no-op; at tiny T it saves compute.

In [ ]:
cfg = fant3_50m(); cfg.mor_adaptive_depth = True
m = FANT3Model(cfg).eval()
for T in (2, 4, 16, 64):
    ids = torch.randint(0, cfg.vocab_size, (1, T))
    with torch.no_grad():
        out = m(ids, targets=ids)
    import math as _math
    k_cap = min(cfg.n_recursion_depths, max(1, int(_math.log2(max(T,1)))+1))
    print(f'T={T:3d}  expected k_cap={k_cap}  forward loss={out["loss"].item():.3f}')
print('PASS (all forwards stable)')

## 5 · RDT K-extrapolation

Set `model.mor.inference_k_override = K` at eval time to push beyond trained `max_depth`. Ignored in training mode.

In [ ]:
cfg = fant3_50m(); m = FANT3Model(cfg).eval()
ids = torch.randint(0, cfg.vocab_size, (1, 32))
for K in (None, 4, 16):
    m.mor.inference_k_override = K
    with torch.no_grad():
        l = m(ids, targets=ids)['loss'].item()
    print(f'inference_k_override={K!s:>4} → loss={l:.4f}')
m.train()
m.mor.inference_k_override = 16  # should be ignored
l_train = m(ids, targets=ids)['loss'].item()
print(f'train-mode (override ignored) → loss={l_train:.4f}')
print('PASS (overrides only fire in eval, no NaN at K=16)')

## 6 · ISRM contractive decay

`cfg.mor_isrm_contractive = True` replaces `current ← next_state` with `current ← current + α_k (next_state − current)` where α_k = (0.15/(1+0.15k))·0.97^k. Strictly contractive → Banach fixed-point, so K=16 inference stays stable even with untrained weights.

In [ ]:
cfg = fant3_50m(); cfg.mor_isrm_contractive = True
m = FANT3Model(cfg).eval()
ids = torch.randint(0, cfg.vocab_size, (1, 32))
for k in range(1, 6):
    ak = (0.15 / (1.0 + 0.15*k)) * (0.97**k)
    print(f'  alpha_k at k={k}: {ak:.4f}')
m.mor.inference_k_override = 16
with torch.no_grad():
    l = m(ids, targets=ids)['loss'].item()
print(f'ISRM + K=16 extrapolation → loss={l:.4f} (must be finite)')
assert l == l and l < 1e6
print('PASS')

## 7 · Area-law attention window (Ryu-Takayanagi)

`cfg.masa_area_law_window = True` replaces full causal with a sliding causal window of size ⌊√T⌋+1. Verify mask shape directly and that forward runs clean.

In [ ]:
import math
T = 36
msk = MASAAttention._area_law_mask(T, 'cpu', torch.float32)
open_row_mid = (msk[20] == 0).sum().item()
open_row_early = (msk[5] == 0).sum().item()
window = int(math.sqrt(T)) + 1
print(f'T={T}  window={window}  row20 open={open_row_mid}  row5 open={open_row_early}')
assert open_row_mid == window + 1 and open_row_early == 6
cfg = fant3_50m(); cfg.masa_area_law_window = True
m = FANT3Model(cfg).eval()
ids = torch.randint(0, cfg.vocab_size, (1, T))
with torch.no_grad():
    l = m(ids, targets=ids)['loss'].item()
print(f'area-law forward loss={l:.4f}')
print('PASS')

## 8 · Litim compact-support LR scheduler

Compare cosine vs Litim shapes. Both share the linear warmup; Litim stays higher mid-run and decays faster at the end.

In [ ]:
W, T = 100, 1000
steps = list(range(0, T+1, 20))
cos = [cosine_schedule(s, W, T) for s in steps]
lit = [litim_schedule(s, W, T) for s in steps]
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(7, 3.2))
    plt.plot(steps, cos, label='cosine', lw=2)
    plt.plot(steps, lit, label='litim',  lw=2)
    plt.axvline(W, ls='--', c='grey', alpha=0.6, label='warmup end')
    plt.xlabel('step'); plt.ylabel('LR multiplier'); plt.legend(); plt.title('Cosine vs Litim compact-support')
    plt.tight_layout(); plt.show()
except Exception as e:
    print('plot skipped:', e)
for s in (0, W, W + (T-W)//2, T):
    print(f'step={s:4d}  cosine={cosine_schedule(s,W,T):.3f}  litim={litim_schedule(s,W,T):.3f}')
assert litim_schedule(T, W, T) == 0.0
assert litim_schedule(W, W, T) == 1.0
print('PASS')

## 9 · Diagonal-EF Fisher preconditioner on router grads

`precondition_router_grads_(model, fisher_state)` mutates `megapool_proj` and `level_proj` grads in place, dividing by `sqrt(EMA(g²)) + ε`. Silent no-op if the training loop isn’t back-propagating through router probs (see the `z_loss` caveat in MEMORY).

In [ ]:
cfg = fant3_50m()
router = MatryoshkaRouter(cfg)
x = torch.randn(2, 16, cfg.dim)
r = router(x)
# Use the router's own auxiliary losses (z_loss + FEP KL) to populate grads
loss = router.z_loss(r['mp_logits'], r['lv_logits']) + router.fep_kl_prior()
# ensure both Linear layers get grads: add a small term touching probs
loss = loss + r['mp_probs'].sum() + r['lv_probs'].sum()
loss.backward()
g_before = {n: p.grad.clone() for n,p in router.named_parameters() if p.grad is not None}
fisher = {}
touched = precondition_router_grads_(router, fisher)
print(f'touched={touched}  fisher_keys={list(fisher.keys())}')
for n in g_before:
    after = router.get_parameter(n).grad
    print(f'  {n}: |g|_before={g_before[n].abs().mean().item():.2e}  |g|_after={after.abs().mean().item():.2e}')
assert touched == 2
print('PASS')

## 10 · FSS scale-ladder collapse

`scripts/fss_collapse.py` fits CE(N, D) = N⁻α · g(D·N⁻β) by minimising the log-log residual of the collapse. Synthetic toy ladder (true α=0.1, β=0.2) should recover α≈0.10, β≈0.21.

In [ ]:
import subprocess, re
r = subprocess.run([sys.executable, 'scripts/fss_collapse.py', '--predict-1b'],
                   capture_output=True, text=True)
print(r.stdout); print(r.stderr)
assert r.returncode == 0, f'fss_collapse.py exited with {r.returncode}'
m = re.search(r'FSS fit:\s*alpha=([\d.]+)\s+beta=([\d.]+)', r.stdout)
assert m, f'Could not parse FSS fit line in stdout:
{r.stdout}'
alpha, beta = float(m.group(1)), float(m.group(2))
print(f'parsed: alpha={alpha:.4f}  beta={beta:.4f}  (truth 0.10, 0.20)')
assert abs(alpha - 0.10) < 0.02, f'alpha={alpha} too far from synthetic truth 0.10'
assert abs(beta  - 0.20) < 0.05, f'beta={beta} too far from synthetic truth 0.20'
print('PASS (recovered synthetic truth within tolerance)')

## 11 · Integration: all flags on, 50m forward + backward + 1 opt step

Turns every new flag to its ‘active’ setting, runs a fresh 50m model through one training step on GPU (if available), verifies no NaN and that every diagnostic surfaces.

In [ ]:
cfg = fant3_50m()
# Enable the full CERN-inspired opt-in set
cfg.mor_lti_injection_enabled  = True
cfg.mor_lti_apollonian_channel = True
cfg.spinor_apollonian_enabled  = True
cfg.mor_adaptive_depth         = True
cfg.mor_isrm_contractive       = True
cfg.masa_area_law_window       = True

device = 'cuda' if torch.cuda.is_available() else 'cpu'
m = FANT3Model(cfg).to(device)
opt = torch.optim.AdamW(m.parameters(), lr=1.5e-4)
B, T = 2, 64
ids = torch.randint(0, cfg.vocab_size, (B, T), device=device)
out = m(ids, targets=ids, store_to_memory=True)
# Also include routing aux losses so Fisher precond has something to rescale
aux = 0.0
if out['router_infos']:
    for ri in out['router_infos']:
        aux = aux + ri['mp_probs'].sum() * 0.0 + ri['lv_probs'].sum() * 0.0  # touch graph
total = out['loss'] + aux
total.backward()
fisher = {}
n_touched = precondition_router_grads_(m, fisher)
opt.step(); opt.zero_grad()
print('integration step:')
print(f'  device        : {device}')
print(f'  loss          : {float(out["loss"]):.4f}  (finite={bool(torch.isfinite(out["loss"]).item())})')
print(f'  fisher touched: {n_touched}  params')
print(f'  memory items  : alpha={int(m.memory.alpha_count)}  beta={int(m.memory.beta_count)}')
stats = m.memory.get_stats()
for k in ('chirality_balance', 'pq_overlap_mean', 'pq_bimodality', 'chsh_S'):
    print(f'  memory.{k:20s}= {stats[k]:+.3f}')
if out['router_infos']:
    rep = float(out['router_infos'][0]['mp_replicon'])
    print(f'  replicon (block 0): {rep:+.3f}')
assert torch.isfinite(out['loss']).item()
print('\nALL TESTS PASS')

---
## Summary

- 11 opt-in features validated end-to-end.
- Default-off flags mean existing training runs are bit-compatible unless you explicitly turn something on.
- To adopt in the 1B flagship run: set the flags in a cfg patch before constructing the model.
- FSS collapse: swap the synthetic toy for real ladder JSON to plan the 1B token budget.